In [7]:
import pandas as pd

df = pd.read_csv("civicdex_adapted.csv")

print(df.columns)

print(df.info())

Index(['request_id', 'raw_text', 'normalized_text', 'english_gloss',
       'language_type', 'intent', 'category', 'department', 'urgency',
       'severity', 'locality', 'district', 'source_type', 'split'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   request_id       660 non-null    object
 1   raw_text         660 non-null    object
 2   normalized_text  660 non-null    object
 3   english_gloss    660 non-null    object
 4   language_type    660 non-null    object
 5   intent           660 non-null    object
 6   category         660 non-null    object
 7   department       660 non-null    object
 8   urgency          660 non-null    object
 9   severity         660 non-null    object
 10  locality         660 non-null    object
 11  district         660 non-null    object
 12  source_type      660 non-null 

In [8]:
df = df[['english_gloss', 'department', 'urgency']]

In [9]:
print(df.isnull().sum())

english_gloss    0
department       0
urgency          0
dtype: int64


In [10]:
df.dropna(inplace=True)

In [11]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mathavan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mathavan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Mathavan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [12]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+|[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [13]:
df['cleaned_text'] = df['english_gloss'].apply(clean_text)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)

X = tfidf.fit_transform(df['cleaned_text'])

In [15]:
y_urgency = df['urgency']

In [16]:
from sklearn.model_selection import train_test_split

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X,
    y_urgency,
    test_size=0.2,
    random_state=42
)

In [17]:
from sklearn.svm import LinearSVC

urgency_model = LinearSVC(class_weight='balanced')

urgency_model.fit(X_train_u, y_train_u)

LinearSVC(class_weight='balanced')

In [18]:
urgency_predictions = urgency_model.predict(X_test_u)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test_u, urgency_predictions))

print()

print(classification_report(y_test_u, urgency_predictions))

Accuracy: 0.7727272727272727

              precision    recall  f1-score   support

        high       0.82      0.82      0.82        56
         low       0.68      0.75      0.71        20
      medium       0.76      0.73      0.75        56

    accuracy                           0.77       132
   macro avg       0.75      0.77      0.76       132
weighted avg       0.77      0.77      0.77       132



In [20]:
import pickle

pickle.dump(urgency_model, open("urgency_model.pkl", "wb"))